# 04. Калькулятор плана: PERT, критический путь, загрузка, окупаемость

**ЛР 3, приложение А.** Ноутбук проверяет ручные расчёты шагов 2–8: трёхточечные оценки, критический путь, календарные даты, загрузку по спринтам и окупаемость. В конце он выводит готовый код диаграммы Ганта для `docs/schedule.md`.

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/04_plan_calculator.ipynb)

Чтобы кнопка открывала ваш ноутбук, замените в ссылке `OWNER/REPO` на владельца и имя своего репозитория. Ноутбуку не нужны данные и Google Диск: всё вводится в ячейках ниже. Запуск: **Среда выполнения → Выполнить все**.

По умолчанию таблицы заполнены примером AquaMap из задания. Замените их своими: работы, оценки, зависимости, исполнители, даты и параметры окупаемости.

Ноутбук проверяет арифметику, но не принимает решения за команду. Зависимости, оценки и выравнивание загрузки — ваша работа.

In [ ]:
import math
from datetime import date, timedelta
import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.1f}".replace(",", " "))

## 1. Исходные данные

**Работы.** Для каждой работы укажите:
- `code` — короткий идентификатор латиницей (используется в зависимостях и в Mermaid);
- `wbs`, `name` — код и название пакета WBS;
- `O`, `M`, `P` — оптимистичная, наиболее вероятная и пессимистичная оценки в часах;
- `days` — длительность в рабочих днях;
- `after` — коды предшественников через пробел (пусто, если работа начинается сразу);
- `who` — исполнители через запятую. Если исполнителей несколько, часы делятся поровну.

**Календарь.** Дата начала, праздники и спринты: имя, первый и последний день, ёмкость одного участника в часах.

In [ ]:
TASKS = [
    # code, wbs,  name,                                     O,  M,  P, days, after,   who
    ("A", "5.1", "Окружение и автопроверки",                 4,  6, 12,  5, "",        "Иван"),
    ("B", "1.5", "Проверка полноты и уточнение застройки",   4,  8, 16,  9, "",        "Олег"),
    ("C", "2.1", "Показатели по районам и школы",            5,  8, 14,  9, "",        "Анна"),
    ("F", "3.1", "Веб-карта: слой точек",                    6, 10, 20, 12, "A",       "Иван"),
    ("D", "2.2", "Доступность 500 м",                        6,  9, 18, 10, "B",       "Олег"),
    ("E", "2.3", "Парки без воды",                           3,  5,  9,  5, "B",       "Анна"),
    ("G", "3.2", "Картограмма районов",                      4,  6, 12,  5, "F C D",   "Олег"),
    ("H", "4.1", "Рекомендации по размещению",               6,  8, 14,  6, "C D E",   "Анна"),
    ("I", "5.2", "Публикация на GitHub Pages",               3,  5, 12,  5, "G",       "Иван"),
    ("J", "4.2", "Выгрузки для департамента",                2,  3,  6,  3, "H",       "Иван"),
    ("K", "6.3", "Итоговый отчёт и защита",                  8, 12, 20,  5, "I J",     "Анна, Олег, Иван"),
]

START = date(2026, 10, 26)               # первый рабочий день периода
HOLIDAYS = {date(2026, 11, 4)}           # нерабочие праздничные дни

SPRINTS = [  # имя, первый день, последний день, ёмкость одного участника на продукт, ч
    ("Sprint 1", date(2026, 10, 26), date(2026, 11, 6), 10),
    ("Sprint 2", date(2026, 11, 9),  date(2026, 11, 20), 10),
    ("Sprint 3", date(2026, 11, 23), date(2026, 12, 4), 10),
    ("Sprint 4", date(2026, 12, 7),  date(2026, 12, 11), 5),
]

df = pd.DataFrame(TASKS, columns=["code", "wbs", "name", "O", "M", "P", "days", "after", "who"]).set_index("code")
df["after"] = df["after"].str.split()
df["who"] = df["who"].str.split(r",\s*")
unknown = {p for preds in df["after"] for p in preds} - set(df.index)
assert not unknown, f"В зависимостях указаны несуществующие работы: {unknown}"
df[["wbs", "name", "O", "M", "P", "days"]]

## 2. Трёхточечная оценка (PERT)

\(E = (O + 4M + P) / 6\), \(\sigma = (P - O) / 6\). Неопределённость проекта: \(\sigma_{проекта} = \sqrt{\sum \sigma_i^2}\).

In [ ]:
df["E"] = (df["O"] + 4 * df["M"] + df["P"]) / 6
df["sigma"] = (df["P"] - df["O"]) / 6
total_E, total_M = df["E"].sum(), df["M"].sum()
sigma = math.sqrt((df["sigma"] ** 2).sum())
print(f"Сумма M: {total_M:.1f} ч · ожидаемая трудоёмкость E: {total_E:.1f} ч · σ проекта: {sigma:.1f} ч")
print(f"Уровень 85%: {total_E + 1.04 * sigma:.1f} ч · уровень 95%: {total_E + 1.645 * sigma:.1f} ч")
df[["wbs", "name", "O", "M", "P", "E", "sigma"]]

## 3. Критический путь (CPM)

Прямой проход: ES = максимум EF предшественников, EF = ES + длительность. Обратный проход: LF = минимум LS последователей, LS = LF − длительность. Резерв = LS − ES. Дни отсчитываются от нуля.

In [ ]:
order, seen = [], set()
def visit(k, stack=()):
    if k in seen: return
    if k in stack: raise ValueError(f"Цикл в зависимостях: {' → '.join(stack + (k,))}")
    for p in df.at[k, "after"]: visit(p, stack + (k,))
    seen.add(k); order.append(k)
for k in df.index: visit(k)

ES, EF = {}, {}
for k in order:
    ES[k] = max((EF[p] for p in df.at[k, "after"]), default=0)
    EF[k] = ES[k] + df.at[k, "days"]
T = max(EF.values())
LS, LF = {}, {}
for k in reversed(order):
    succ = [s for s in df.index if k in df.at[s, "after"]]
    LF[k] = min((LS[s] for s in succ), default=T)
    LS[k] = LF[k] - df.at[k, "days"]
df["ES"], df["EF"], df["LS"], df["LF"] = pd.Series(ES), pd.Series(EF), pd.Series(LS), pd.Series(LF)
df["slack"] = df["LS"] - df["ES"]
critical = [k for k in order if df.at[k, "slack"] == 0]
print(f"Срок проекта: {T} рабочих дней")
print("Критический путь:", " → ".join(f"{df.at[k, 'wbs']} {df.at[k, 'name']}" for k in critical))
df.loc[order, ["wbs", "name", "days", "ES", "EF", "LS", "LF", "slack"]]

## 4. Календарные даты

Рабочие дни переводятся в даты без выходных и праздников из `HOLIDAYS`.

In [ ]:
workdays, d = [], START
while len(workdays) < T + 60:
    if d.weekday() < 5 and d not in HOLIDAYS:
        workdays.append(d)
    d += timedelta(days=1)
df["start"] = [workdays[df.at[k, "ES"]] for k in df.index]
df["finish"] = [workdays[df.at[k, "EF"] - 1] for k in df.index]
print("Окончание работ:", workdays[T - 1].strftime("%d.%m.%Y"))
df.loc[order, ["wbs", "name", "start", "finish", "slack"]]

## 5. Загрузка по спринтам

Часы работы распределяются по дням её выполнения поровну, а между несколькими исполнителями — поровну. Процент — доля от ёмкости участника в спринте.

In [ ]:
def sprint_of(day):
    for name, first, last, _ in SPRINTS:
        if first <= day <= last:
            return name
    return "вне спринтов"

load = {}
for k in df.index:
    per_day = df.at[k, "E"] / df.at[k, "days"]
    people = df.at[k, "who"]
    for i in range(df.at[k, "ES"], df.at[k, "EF"]):
        for person in people:
            key = (person, sprint_of(workdays[i]))
            load[key] = load.get(key, 0) + per_day / len(people)

people = sorted({p for ps in df["who"] for p in ps})
names = [s[0] for s in SPRINTS] + (["вне спринтов"] if any(k[1] == "вне спринтов" for k in load) else [])
hours = pd.DataFrame([[load.get((p, s), 0) for s in names] for p in people], index=people, columns=names)
cap = {s[0]: s[3] for s in SPRINTS}
percent = hours.apply(lambda col: col / cap[col.name] * 100 if col.name in cap else col * float("nan"))
print("Часы:"); display(hours.assign(Всего=hours.sum(axis=1)))
print("Загрузка, % от ёмкости (больше 110% — перегрузка):"); display(percent.round(0))

## 6. Код диаграммы Ганта для docs/schedule.md

Скопируйте вывод ячейки в блок кода mermaid и добавьте вехи (релизы, защиту) и буфер проекта. Работы распределены по секциям спринтов по дате начала, критические отмечены `crit`. Двоеточия в названиях работ заменяются на тире: в Mermaid двоеточие отделяет название от параметров.

In [ ]:
BUFFER_DAYS = 4                 # буфер проекта после последней работы, рабочих дней
DEFENSE = date(2026, 12, 18)    # финальная веха

lines = ["gantt", "    title Базовый календарный план", "    dateFormat YYYY-MM-DD", "    axisFormat %d.%m",
         "    excludes weekends" + "".join(f", {h.isoformat()}" for h in sorted(HOLIDAYS))]
current = None
for k in sorted(order, key=lambda k: (df.at[k, "ES"], order.index(k))):
    section = sprint_of(df.at[k, "start"])
    if section != current:
        lines.append(f"    section {section}"); current = section
    tag = "crit, " if df.at[k, "slack"] == 0 else ""
    when = f"after {' '.join(c.lower() for c in df.at[k, 'after'])}" if df.at[k, "after"] else df.at[k, "start"].isoformat()
    label = f"{df.at[k, 'wbs']} {df.at[k, 'name']}".replace(":", " —").replace("#", "№")
    lines.append(f"    {label} :{tag}{k.lower()}, {when}, {df.at[k, 'days']}d")
last = max(order, key=lambda k: df.at[k, "EF"]).lower()
lines.append(f"    Буфер проекта :buf, after {last}, {BUFFER_DAYS}d")
lines.append(f"    Защита :milestone, crit, m_end, {DEFENSE.isoformat()}, 0d")
print("\n".join(lines))

## 7. Окупаемость: TCO, NPV, ROI

Все цифры условные, допущения запишите в `docs/budget.md`.

In [ ]:
INVESTMENT = 250_000      # базовый бюджет, руб.
BENEFIT = 150_000         # выгода заказчика в год, руб.
SUPPORT = 18_000          # поддержка в год, руб.
RATE = 0.20               # ставка дисконтирования
YEARS = 3

def npv(benefit):
    cf = benefit - SUPPORT
    return -INVESTMENT + sum(cf / (1 + RATE) ** t for t in range(1, YEARS + 1))

cf = BENEFIT - SUPPORT
tco = INVESTMENT + YEARS * SUPPORT
print(f"TCO за {YEARS} года: {tco:,.0f} руб.".replace(",", " "))
print(f"Годовой чистый поток: {cf:,.0f} руб.".replace(",", " "))
print(f"NPV: {npv(BENEFIT):,.0f} руб.".replace(",", " "))
print(f"ROI: {(YEARS * BENEFIT - tco) / tco:.0%}")
print(f"Срок окупаемости: {INVESTMENT / cf:.1f} года" if cf > 0 else "Проект не окупается: поток не положительный")

annuity = sum(1 / (1 + RATE) ** t for t in range(1, YEARS + 1))
breakeven = INVESTMENT / annuity + SUPPORT
print(f"NPV = 0 при выгоде {breakeven:,.0f} руб. в год".replace(",", " "))
pd.DataFrame({"Выгода в год, руб.": b, "NPV, руб.": round(npv(b))} for b in range(90_000, 210_001, 30_000))